# Lab: Hand-crafted Voice Features and Speech Classification
**Duration:** about 90 minutes.

In the PStA you will classify speakers as healthy (HC), mildly cognitively impaired (MCI) or demented (DEM) from their speech.
Before we touch that data, we learn the toolbox on a public, well-understood corpus: **EmoDB**, the Berlin Database of Emotional Speech.
Emotion recognition and pathological speech analysis are both **paralinguistic** tasks.
We do not care *what* is said, but *how* it is said: pitch, loudness, timing and voice quality.
The features you compute today (F0, jitter, shimmer, HNR, CPP, pauses, eGeMAPS) are exactly the ones used in clinical voice research on Parkinson's disease, dysphonia and dementia.
The evaluation pitfalls you will run into (speaker leakage, confounders, tuning on the test set) are the ones listed on the PStA slides.

## Learning goals
After this lab you can
- explain what F0, intensity, jitter, shimmer, HNR, CPP and formants measure, physiologically and computationally,
- extract them with Praat (via `parselmouth`) and openSMILE, and explain the difference between low-level descriptors and functionals,
- implement a measure like jitter yourself and check it against a reference implementation,
- build a classification pipeline with `scikit-learn` that is evaluated **speaker-independently**,
- explain why random cross-validation is too optimistic for speech data and what speaker normalisation does,
- transfer these ideas to the PARLO Dementia Corpus.

## How to work
Most of the code is given, because the lab is about *understanding*, not about programming.
Wherever you see `# TODO`, a short piece of code is missing, and it is always the one that carries the key idea.
For every experiment the routine is: **predict → run → look → listen → explain**.
Write down your prediction *before* you run a cell, then compare.

| Part | Topic | Time |
|---|---|---|
| 0 | Setup and data | 10 min |
| 1 | Prosody: F0, intensity, timing | 20 min |
| 2 | Voice quality: jitter, shimmer, HNR, CPP, formants | 20 min |
| 3 | Low-level descriptors, functionals and openSMILE | 10 min |
| 4 | Classification done right | 25 min |
| 5 | Transfer to the PStA | 5 min |

# Part 0: Setup and Data (10 min)

In [ ]:
# run if needed (the course uv environment already contains everything):
#!pip install audb opensmile praat-parselmouth librosa scikit-learn pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import parselmouth                      # Python interface to Praat
from parselmouth.praat import call      # call any Praat command by its name
from IPython.display import display

from slp_lab_utils import SR, EMOTIONS, load_emodb, load_audio, play, cached, speaker_znorm

## The data: EmoDB
- 10 German actors (5 female, 5 male, aged 21 to 35) each read the **same 10 sentences** in **7 emotions**: neutral, anger, boredom, disgust, fear, happiness and sadness.
- Recorded in an anechoic chamber at 16 kHz, together with a **laryngograph** (electroglottograph, EGG).
  The laryngograph measures the electrical impedance across the larynx, so it records the opening and closing of the vocal folds directly, without the vocal tract on top.
  We use it as a near ground truth for F0.
- A listening test rated every utterance; only utterances recognised by most listeners are in the "gold standard" tables.
- We use the **official split** shipped with `audb`: `emotion.categories.train.gold_standard` and `emotion.categories.test.gold_standard`.
- There is **no development set**.
  All model choices therefore have to be made by cross-validation *inside* the training set, and the test set is used **exactly once** at the end.

The first call downloads about 40 MB to `~/audb`, later calls load from there.

In [ ]:
train, test = load_emodb()
display(train.head())
print(len(train), 'train utterances,', len(test), 'test utterances')

## Task 0.1: Know your data

In [ ]:
train_speakers = set(train.speaker)
test_speakers = set(test.speaker)
overlap = train_speakers & test_speakers
print('train speakers:', sorted(train_speakers))
print('test speakers: ', sorted(test_speakers))
print('overlap:       ', overlap)

counts = pd.crosstab(pd.concat([train, test]).emotion, ['train'] * len(train) + ['test'] * len(test))
display(counts)
display(pd.crosstab([train.speaker, train.gender], train.emotion))  # utterances per training speaker and emotion

**Questions**
1. Is the split speaker-independent? Why does this matter for a model that is supposed to work on *new* people (patients)?
2. The classes are imbalanced. Why is accuracy a bad metric here, and what does UAR (unweighted average recall) do instead?
3. Look at the table per training speaker. Which speaker-emotion combinations are (almost) missing? What does that mean when we later validate on one speaker at a time?
4. Every emotion uses the same 10 sentences. What does this mean for text-based (linguistic) features on EmoDB, and how is the PARLO corpus different?

**Solution:**
1. Yes: train has speakers 3, 8, 9, 10, 11, 13 and test has 12, 14, 15, 16, so the overlap is empty.
   A model sees a voice, a speaking style and a microphone distance per speaker.
   If the same speaker is in train and test, the model can partly recognise the *speaker* instead of the emotion (or the disease), and the score is too optimistic for new people.
2. Anger is 24 % of the training data and disgust only 7 %.
   A classifier that ignores disgust loses little accuracy.
   UAR averages the recall of every class with equal weight (sklearn: `balanced_accuracy_score`), so chance level is 1/7 ≈ 14 % for 7 classes, independent of the class distribution.
   The PStA uses UAR for the same reason.
3. Speaker 8 has no disgust utterance at all, and speakers 3 and 10 have only one.
   When speaker 8 is the validation fold, the recall of disgust is undefined, so the fold's UAR is computed over 6 classes only (sklearn warns about this), and single utterances decide the recall of a class.
   Per-fold scores are therefore noisy, and that is normal for small clinical datasets as well.
4. The text carries no information about the emotion, because it is identical across classes, so here only *how* something is said counts.
   In the PARLO corpus, *what* people say (word-finding problems, information content, recall) is highly informative, so there acoustic and linguistic features complement each other.

Listen to one speaker saying the same sentence in all seven emotions. Close your eyes and try to name them.

In [ ]:
same_sentence = train[(train.speaker == 3) & (train.sentence == 'a01')].sort_values('emotion')
print(same_sentence.text.iloc[0])
for _, row in same_sentence.iterrows():
    print(row.emotion)
    display(play(load_audio(row.file)))

# Part 1: Prosody: F0, Intensity and Timing (20 min)

**Prosody** is the melody, loudness and rhythm of speech.
It is the classical carrier of emotion, and in dementia research it is linked to reduced variability ("flat" speech), slower speech and more pauses.

## Fundamental frequency (F0)
- In voiced sounds the vocal folds open and close periodically.
  The rate of these glottal cycles is the **fundamental frequency F0**, measured in Hz (recall the source-filter model: F0 belongs to the *source*).
- **Pitch** is the *perceived* height of a tone. It is mainly determined by F0, but it is a perceptual quantity.
- Typical F0 of adult speakers: roughly 85-155 Hz for male and 165-255 Hz for female voices.
  Within an utterance F0 moves a lot (intonation).
- Only **voiced** frames have an F0. For unvoiced sounds (/s/, /f/) and pauses the pitch tracker reports "no F0".
- We perceive pitch **logarithmically**: an octave (doubling of F0) sounds like the same step whether it is 100→200 Hz or 200→400 Hz.
  Therefore we use **semitones** (12 per octave) relative to a reference frequency $f_\text{ref}$:
  $$\text{st} = 12 \log_2 \frac{F_0}{f_\text{ref}}$$
  On this scale a male and a female voice that move "the same" melodically have the same standard deviation.
- Pitch trackers make typical errors: **octave errors** (half or double the true F0) and **voicing errors** (unvoiced frames detected as voiced and vice versa).
  Creaky voice, breathy voice and noise make both more likely, and these voice qualities are frequent in older and pathological speakers.

We compare two pitch trackers:
- **Praat** (autocorrelation method, Boersma 1993): finds the lag at which the signal is most similar to a shifted copy of itself.
- **pYIN** in librosa (probabilistic YIN): a difference-function method with an HMM that smooths the voiced/unvoiced decision over time.

Both are compared with F0 computed on the **laryngograph** signal, which has no vocal tract filter on top and is therefore much easier to track.

In [ ]:
row = train[(train.speaker == 3) & (train.sentence == 'a01') & (train.emotion == 'anger')].iloc[0]
y = load_audio(row.file)
F0_MIN, F0_MAX = 75, 600   # search range in Hz, wide enough for male and female voices

# Praat
snd = parselmouth.Sound(y, sampling_frequency=SR)
pitch = snd.to_pitch_ac(pitch_floor=F0_MIN, pitch_ceiling=F0_MAX)
f0_praat = pitch.selected_array['frequency']          # 0 Hz = unvoiced
f0_praat[f0_praat == 0] = np.nan
t_praat = pitch.xs()

# laryngograph (EGG) reference, analysed with the same Praat settings
egg = parselmouth.Sound(row.laryngo)
pitch_egg = egg.to_pitch_ac(pitch_floor=F0_MIN, pitch_ceiling=F0_MAX)
f0_egg = pitch_egg.selected_array['frequency']
f0_egg[f0_egg == 0] = np.nan

# pYIN
f0_pyin, voiced_flag, voiced_prob = librosa.pyin(y, fmin=F0_MIN, fmax=F0_MAX, sr=SR, hop_length=160)
t_pyin = librosa.times_like(f0_pyin, sr=SR, hop_length=160)

fig, ax = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
D = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=1024, hop_length=160)), ref=np.max)
librosa.display.specshow(D, sr=SR, hop_length=160, x_axis='time', y_axis='linear', ax=ax[0], cmap='magma', vmin=-80)
ax[0].set_ylim(0, 1000)
ax[0].set_title(f'narrowband spectrogram, speaker {row.speaker}, {row.emotion}: the harmonics are multiples of F0')
ax[1].plot(pitch_egg.xs(), f0_egg, 'k', lw=4, alpha=0.3, label='laryngograph (reference)')
ax[1].plot(t_praat, f0_praat, '.', label='Praat (microphone)')
ax[1].plot(t_pyin, f0_pyin, '.', label='pYIN (microphone)')
ax[1].set_ylabel('F0 [Hz]')
ax[1].set_xlabel('time [s]')
ax[1].legend()
plt.tight_layout()
display(play(y))

**Questions**
1. Where do the trackers disagree with the reference: in the middle of vowels, or at the beginning and end of voiced segments? Why there?
2. Do you see octave jumps? Which tracker produces the smoother contour, and why (think of the HMM in pYIN)?
3. Look at the narrowband spectrogram: how could you read F0 directly from it?

**Solution:**
1. Mostly at voicing onsets and offsets: there the glottal cycles are irregular and short, the analysis window contains voiced and unvoiced parts, and the voicing decision is uncertain.
   In the middle of vowels all three agree within a few Hz.
2. Single frames with half or double the reference F0 are octave errors, typically in creaky or very loud parts.
   pYIN is smoother because its HMM penalises jumps between successive frames, so isolated outliers are suppressed.
3. The lowest harmonic (the lowest horizontal line) lies at F0, and the distance between neighbouring harmonics is also F0.

## Intensity, pauses and speaking rate
- **Intensity** is the energy of the signal in short windows, in dB.
  It correlates with perceived loudness and with vocal effort, and it is a strong cue for **arousal** (anger vs. boredom).
  Absolute values depend on the microphone distance and gain, so they are only comparable within one recording setup, and relative measures (variability, range) are safer.
- **Pauses** are stretches whose intensity is far below the loudest part of the utterance (here: more than 25 dB).
  More and longer pauses are among the most consistent findings in the speech of people with AD and MCI (word-finding, planning).
- **Voiced ratio** is the share of frames that have an F0.
- **Speaking rate**: because all EmoDB speakers read the same sentences, the **duration** of an utterance is a direct (inverse) measure of the speaking rate.
  In free speech (PARLO picture description) you need the number of syllables or words per second, for example from an ASR transcript.

## Task 1.1: Prosodic features for one utterance
Complete the semitone conversion and the robust F0 range.
We use the 5th and 95th percentile instead of min and max, because single octave errors would otherwise dominate the range.

In [ ]:
def prosodic_features(snd, f0_ref=100.0):
    pitch = snd.to_pitch_ac(pitch_floor=F0_MIN, pitch_ceiling=F0_MAX)
    f0 = pitch.selected_array['frequency']
    f0_voiced = f0[f0 > 0]
    st = 12 * np.log2(f0_voiced / f0_ref)
    f0_range = np.percentile(st, 95) - np.percentile(st, 5)

    intensity = snd.to_intensity(minimum_pitch=F0_MIN).values[0]
    speech = intensity > intensity.max() - 25      # frames within 25 dB of the loudest frame
    return {
        'f0_mean_st': st.mean(),
        'f0_std_st': st.std(),
        'f0_range_st': f0_range,
        'voiced_ratio': np.mean(f0 > 0),
        'intensity_mean_db': intensity[speech].mean(),
        'intensity_std_db': intensity[speech].std(),
        'pause_ratio': 1 - speech.mean(),
        'duration_s': snd.duration,
    }

prosodic_features(snd)

## Task 1.2: Prosody across emotions and speakers
The following cell computes the features for all training utterances and plots the mean F0 per emotion, split by gender.

In [ ]:
pros = pd.DataFrame([prosodic_features(parselmouth.Sound(f)) for f in train.file])
pros[['emotion', 'speaker', 'gender']] = train[['emotion', 'speaker', 'gender']].values

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, feat in zip(ax, ['f0_mean_st', 'f0_std_st', 'duration_s']):
    pros.pivot_table(index='emotion', columns='gender', values=feat, observed=True).loc[EMOTIONS].plot.bar(ax=a)
    a.set_title(feat)
plt.tight_layout()

**Questions**
1. Which emotions have high and which low F0? Group the emotions by **arousal** (activation).
2. Compare the difference between male and female speakers with the difference between emotions. What does this mean for a classifier that sees raw `f0_mean_st`?
3. Which emotion is spoken most slowly? Does this fit your listening impression from Part 0?
4. Transfer: in the PARLO corpus, the gender mix differs slightly between HC, MCI and DEM. Why is that a problem for F0-based features?

**Solution:**
1. High F0: anger, happiness, fear and (less) disgust, the high-arousal emotions.
   Low F0: neutral, boredom and sadness, the low-arousal ones.
2. Male and female voices differ by about 10-12 semitones, which is as large as or larger than the effect of emotion.
   Raw F0 therefore separates gender at least as well as emotion, and a classifier will partly learn gender.
   This motivates **speaker (or at least gender) normalisation**, see Part 4.
3. Sadness and boredom are the slowest (longest durations), anger and fear are faster.
4. If one class has more women, then "high F0" becomes a cue for that class even though it only encodes gender: a **confounder**.
   You have to check such effects, for example by normalising per gender or by reporting results per gender.

# Part 2: Voice Quality: Jitter, Shimmer, HNR, CPP and Formants (20 min)

Voice quality describes *how* the vocal folds vibrate, independent of pitch and loudness: modal, breathy, creaky, rough, hoarse.
A healthy voice in a sustained vowel is nearly, but never perfectly, periodic.
Diseases of the larynx, neurological diseases (Parkinson's, ALS), ageing and, in later stages, dementia make the vibration less regular and add turbulent noise.
The following measures quantify this.
They come from clinical voice analysis (e.g. the MDVP software) and are all part of eGeMAPS.

Let $T_i$ be the duration of the $i$-th glottal cycle (period) and $A_i$ its peak amplitude, $N$ the number of cycles.

| Measure | Definition | What it captures | Typical source of high values |
|---|---|---|---|
| **Jitter (local)** | $\frac{\frac{1}{N-1}\sum_{i=1}^{N-1} \lvert T_i - T_{i+1} \rvert}{\frac{1}{N}\sum_{i=1}^{N} T_i}$ | cycle-to-cycle **frequency** perturbation | irregular vocal fold vibration, poor neuromuscular control, creaky voice |
| **Jitter (RAP)** | like local, but $T_i$ compared with the mean of $T_{i-1}, T_i, T_{i+1}$ | same, less sensitive to slow F0 movements (intonation) | as above |
| **Shimmer (local)** | $\frac{\frac{1}{N-1}\sum \lvert A_i - A_{i+1} \rvert}{\frac{1}{N}\sum A_i}$ | cycle-to-cycle **amplitude** perturbation | incomplete glottal closure, lesions, reduced breath support |
| **Shimmer (APQ3)** | amplitude version of RAP (3-point average) | same, smoother | as above |
| **HNR** (harmonics-to-noise ratio) | $10 \log_{10} \frac{E_\text{periodic}}{E_\text{noise}}$ in dB | how much of the energy is periodic | breathiness (air leaking through the glottis), hoarseness |
| **CPP(S)** (smoothed cepstral peak prominence) | height of the cepstral peak at the pitch period above a regression line through the cepstrum, in dB | how clearly the signal has *one* dominant periodicity | dysphonia in general; robust in running speech, therefore the clinical favourite today |
| **Formants F1-F3** | resonance frequencies of the vocal tract (the *filter*) | articulation: F1 ~ jaw opening, F2 ~ tongue front/back | reduced articulation (centralised vowels), vocal tract length |

Some notes:
- Jitter and shimmer are only meaningful for **voiced** stretches with a reliable period detection, so they inherit all the errors of the pitch tracker.
- Clinical norm values (e.g. jitter < 1.04 %, shimmer < 3.81 % in MDVP) are defined for **sustained vowels** ("aaaa" for several seconds).
  In running speech, with intonation and many voicing onsets, values are much higher and only comparable *within* one task and setup.
  In the PARLO battery, the "pataka" repetition and the story reading are the most controlled tasks.
- The **cepstrum** is the spectrum of the log spectrum.
  A periodic signal has evenly spaced harmonics, these ripples in the log spectrum show up as one sharp peak in the cepstrum at "quefrency" $1/F_0$.
  The clearer the harmonics, the higher the peak.

## See and hear perturbations
The cell below synthesises a vowel-like sound: a train of glottal pulses (the source) filtered by two resonances (the vocal tract).
Every pulse gets a random period deviation (jitter) and amplitude deviation (shimmer), and we can add noise (lower HNR).
Change the parameters, look at the cycles and listen.

In [ ]:
from scipy.signal import lfilter

def synthetic_vowel(f0=120, jitter=0.0, shimmer=0.0, noise=0.0, dur=1.5, seed=0):
    rng = np.random.default_rng(seed)
    source, t = np.zeros(int(dur * SR)), 0.0
    while t < dur - 0.02:
        source[int(t * SR)] = 1 + shimmer * rng.standard_normal()   # shimmer: random amplitude deviation
        t += (1 / f0) * (1 + jitter * rng.standard_normal())        # jitter: random period deviation
    source += noise * 0.05 * rng.standard_normal(len(source))       # turbulent noise: lowers the HNR
    # vocal tract filter: two resonances at 700 Hz (F1) and 1200 Hz (F2), roughly an /a/
    for fc, bw in [(700, 80), (1200, 100)]:
        r, theta = np.exp(-np.pi * bw / SR), 2 * np.pi * fc / SR
        source = lfilter([1], [1, -2 * r * np.cos(theta), r ** 2], source)
    return 0.5 * source / np.abs(source).max()

voices = {
    'healthy': synthetic_vowel(),
    'jitter 3 %': synthetic_vowel(jitter=0.03),
    'shimmer 20 %': synthetic_vowel(shimmer=0.2),
    'breathy (noise)': synthetic_vowel(noise=1.0),
}
fig, ax = plt.subplots(len(voices), 1, figsize=(14, 8), sharex=True)
for a, (name, v) in zip(ax, voices.items()):
    a.plot(np.arange(len(v)) / SR, v, lw=0.7)
    a.set_xlim(0.5, 0.6)
    a.set_title(name)
    print(name)
    display(play(v))
ax[-1].set_xlabel('time [s]')
plt.tight_layout()

## Task 2.1: Jitter by hand
Praat's `PointProcess` marks one time point per glottal cycle.
The differences between successive points are the periods $T_i$.
Praat ignores implausible periods: shorter than 0.1 ms, longer than 20 ms (these span a pause), or more than 1.3 times longer or shorter than the neighbouring period.
Compute the local jitter with the formula from the table and compare it with Praat.

In [ ]:
points = call(snd, 'To PointProcess (periodic, cc)', F0_MIN, F0_MAX)
t_cycles = np.array([call(points, 'Get time from index', i + 1) for i in range(call(points, 'Get number of points'))])

fig, ax = plt.subplots(figsize=(14, 3))
t = np.arange(len(y)) / SR
ax.plot(t, y, lw=0.5)
ax.vlines(t_cycles, -1, 1, color='r', lw=0.5)
ax.set_xlim(0.6, 0.7)
ax.set_ylim(-np.abs(y).max(), np.abs(y).max())
ax.set_title('waveform with one red line per detected glottal cycle (zoom: 100 ms)')
ax.set_xlabel('time [s]');

In [ ]:
T = np.diff(t_cycles)                                      # periods in seconds
valid = (T >= 0.0001) & (T <= 0.02)                        # plausible single periods
ratio = np.maximum(T[1:], T[:-1]) / np.minimum(T[1:], T[:-1])
valid_pairs = valid[1:] & valid[:-1] & (ratio <= 1.3)      # neighbouring periods we may compare

jitter_mine = np.mean(np.abs(np.diff(T))[valid_pairs]) / np.mean(T[valid])
jitter_praat = call(points, 'Get jitter (local)', 0, 0, 0.0001, 0.02, 1.3)
print(f'jitter (local): mine = {100 * jitter_mine:.2f} %   Praat = {100 * jitter_praat:.2f} %')
print(f'mean F0 from the periods: {1 / np.mean(T[valid]):.0f} Hz')

## Task 2.2: All hand-crafted features for the whole corpus
The function below collects the prosodic features from Part 1 and the voice quality measures.
The numeric arguments of the Praat commands are the standard values from the Praat voice report: period floor 0.1 ms, period ceiling 20 ms, maximum period factor 1.3 and maximum amplitude factor 1.6.
Extraction takes about 30 s per split, and the result is cached in `~/.cache/slp_labs`.

In [ ]:
def voice_features(path):
    snd = parselmouth.Sound(path)
    feats = prosodic_features(snd)
    points = call(snd, 'To PointProcess (periodic, cc)', F0_MIN, F0_MAX)
    feats['jitter_local'] = call(points, 'Get jitter (local)', 0, 0, 0.0001, 0.02, 1.3)
    feats['jitter_rap'] = call(points, 'Get jitter (rap)', 0, 0, 0.0001, 0.02, 1.3)
    feats['shimmer_local'] = call([snd, points], 'Get shimmer (local)', 0, 0, 0.0001, 0.02, 1.3, 1.6)
    feats['shimmer_apq3'] = call([snd, points], 'Get shimmer (apq3)', 0, 0, 0.0001, 0.02, 1.3, 1.6)
    harmonicity = snd.to_harmonicity_cc(minimum_pitch=F0_MIN)
    feats['hnr_db'] = call(harmonicity, 'Get mean', 0, 0)
    cepstrogram = call(snd, 'To PowerCepstrogram', 60, 0.002, 5000, 50)
    feats['cpps_db'] = call(cepstrogram, 'Get CPPS', 'no', 0.01, 0.001, 60, 330, 0.05, 'parabolic', 0.001, 0,
                            'Straight', 'Robust')
    formants = snd.to_formant_burg(maximum_formant=5500)
    for k in (1, 2, 3):
        feats[f'f{k}_hz'] = call(formants, 'Get mean', k, 0, 0, 'hertz')
    return feats


voice_train = cached('emodb_voice_train', lambda: pd.DataFrame([voice_features(f) for f in train.file]))
voice_test = cached('emodb_voice_test', lambda: pd.DataFrame([voice_features(f) for f in test.file]))
voice_train.describe().T.round(3)

To compare features with very different units (Hz, dB, %), we standardise every feature over the training set (mean 0, standard deviation 1) with `sklearn.preprocessing.StandardScaler` and plot the mean per emotion.

In [ ]:
from sklearn.preprocessing import StandardScaler

z = pd.DataFrame(StandardScaler().fit_transform(voice_train), columns=voice_train.columns)
profile = z.groupby(train.emotion.values, observed=True).mean().loc[EMOTIONS]

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(profile.T, cmap='RdBu_r', vmin=-1.5, vmax=1.5, aspect='auto')
ax.set_xticks(range(len(EMOTIONS)), EMOTIONS, rotation=45)
ax.set_yticks(range(profile.shape[1]), profile.columns)
plt.colorbar(im, label='mean z-score')
ax.set_title('Feature profile per emotion (train)');

**Questions**
1. Does your hand-written jitter match Praat? If it differs slightly, where could the difference come from?
2. Which features separate the **high-arousal** emotions (anger, happiness, fear) from the **low-arousal** ones (boredom, sadness, neutral)?
3. Which pairs of emotions look almost identical in this profile? What does that predict for the confusion matrix later?
4. Sadness has the lowest CPPS and the highest jitter and shimmer. What does that say about the voice quality the actors used?
5. Anger shows a *lower* mean intensity and a *higher* pause ratio than most other emotions. Is anger really quieter, with more pauses? Look at how `prosodic_features` defines both features.
6. Why are jitter and shimmer fragile on short, emotional sentences, and why would you prefer a sustained vowel or the "pataka" task for them in a clinical setting?

**Solution:**
1. Yes, to about 0.01 percentage points.
   Small differences come from details of the period selection, for example how Praat handles the first and last cycle of each voiced stretch.
2. Mean F0 and CPPS are high for anger and happiness and low for neutral, boredom and sadness.
   Sadness additionally stands out by long durations (slow speech) and a low voiced ratio, and boredom by a high HNR and low shimmer (a relaxed, regular voice).
3. Anger and happiness look very similar (high F0, high CPPS), and disgust and fear have weak profiles close to neutral.
   We expect these confusions, because arousal is easy to measure acoustically, but valence (positive vs. negative) is not.
4. The actors used a weak, irregular voice for sadness: the vocal folds vibrate less regularly (jitter, shimmer) and the harmonics are less clear (CPPS).
   The same measures are used to describe dysphonic voices, for example in Parkinson's disease.
5. No, this is an artefact of the feature definition.
   Both features are relative to the *loudest* frame of the utterance: anger has very loud peaks, so more frames fall below "maximum − 25 dB" and count as pauses, and the mean over the remaining frames drops.
   A feature is only what its code computes, so always check that it measures what its name promises before you interpret it, especially before drawing clinical conclusions.
6. Every sentence contains many voicing onsets and offsets and fast intonation changes.
   These change the periods for reasons that have nothing to do with irregular vocal fold vibration, and the pitch tracker makes more errors there.
   In a sustained vowel the only source of perturbation is the voice itself.
   "Pataka" is constrained and similar for all speakers, so the measures are at least comparable between groups.

# Part 3: Low-level Descriptors, Functionals and openSMILE (10 min)

All features so far follow the same two-step recipe:
1. **Low-level descriptors (LLDs)** are computed per frame (e.g. every 10 ms): F0, intensity, MFCCs, ...
2. **Functionals** summarise the LLD contour of a whole utterance into one number: mean, standard deviation, percentiles, slope, ...

This turns a variable-length recording into a fixed-length vector that any classical classifier can use.
**openSMILE** (audEERING) implements this recipe for large, standardised feature sets:
- **eGeMAPS v02** (Eyben et al., 2016): 88 functionals of 25 LLDs (frequency, energy, spectral balance and temporal features), chosen by experts for paralinguistics and clinical voice research.
  Small, interpretable, and the standard baseline in the field.
- **ComParE 2016**: 6373 functionals, a brute-force set used as the baseline of the Interspeech ComParE challenges.

Below we also compute MFCC functionals (mean and standard deviation of 20 MFCCs and their deltas) as introduced in the lecture.

In [ ]:
import opensmile

egemaps = opensmile.Smile(feature_set=opensmile.FeatureSet.eGeMAPSv02, feature_level=opensmile.FeatureLevel.Functionals)
compare = opensmile.Smile(feature_set=opensmile.FeatureSet.ComParE_2016, feature_level=opensmile.FeatureLevel.Functionals)

def smile(extractor, files):
    return extractor.process_files(files).reset_index(drop=True)

egemaps_train = cached('emodb_egemaps_train', lambda: smile(egemaps, train.file))
egemaps_test = cached('emodb_egemaps_test', lambda: smile(egemaps, test.file))
compare_train = cached('emodb_compare_train', lambda: smile(compare, train.file))
compare_test = cached('emodb_compare_test', lambda: smile(compare, test.file))
print('eGeMAPS:', egemaps_train.shape, '  ComParE:', compare_train.shape)


def mfcc_functionals(path, n_mfcc=20):
    mfcc = librosa.feature.mfcc(y=load_audio(path), sr=SR, n_mfcc=n_mfcc)
    delta = librosa.feature.delta(mfcc)
    return np.concatenate([mfcc.mean(1), mfcc.std(1), delta.mean(1), delta.std(1)])

mfcc_train = cached('emodb_mfcc_train', lambda: np.stack([mfcc_functionals(f) for f in train.file]))
mfcc_test = cached('emodb_mfcc_test', lambda: np.stack([mfcc_functionals(f) for f in test.file]))

## Task 3.1: Find your features in eGeMAPS
eGeMAPS contains its own jitter, shimmer, HNR and F0 functionals.
List them, and check how strongly our Praat jitter correlates with the eGeMAPS jitter.

In [ ]:
names = [c for c in egemaps_train.columns if any(k in c for k in ['F0semitone', 'jitter', 'shimmer', 'HNR'])]
print('\n'.join(names))

r_jitter = np.corrcoef(voice_train.jitter_local, egemaps_train.jitterLocal_sma3nz_amean)[0, 1]
r_f0 = np.corrcoef(voice_train.f0_mean_st, egemaps_train['F0semitoneFrom27.5Hz_sma3nz_amean'])[0, 1]
print(f'r(jitter) = {r_jitter:.2f}    r(F0 mean) = {r_f0:.2f}')

**Questions**
1. The F0 means correlate almost perfectly, the jitter values much less. Why?
2. eGeMAPS uses 27.5 Hz as the semitone reference and we used 100 Hz. Does this matter for a classifier?

**Solution:**
1. The mean F0 is robust: small differences in pitch tracking average out over hundreds of frames.
   Jitter measures tiny cycle-to-cycle differences, so it depends strongly on the period detection algorithm, on how voiced segments are cut, and on smoothing (`_sma3` means a 3-frame moving average in openSMILE).
   Two "jitter" implementations are not the same feature, so always report which tool and settings you used.
2. No: changing the reference only adds a constant ($12 \log_2 (100/27.5)$ semitones), which disappears after standardisation.

# Part 4: Classification Done Right (25 min)

We now train classifiers with `scikit-learn`.
Three rules keep the evaluation honest:
1. **Everything that is fitted on data goes into a `Pipeline`**: imputation of missing values, scaling and the classifier.
   During cross-validation the whole pipeline is re-fitted on every training fold, so no statistics of the validation fold leak into training.
2. **Split by speaker**: all utterances of one speaker are either in training or in validation.
   With 6 training speakers, `LeaveOneGroupOut` gives 6 folds: train on 5 speakers, validate on the 6th.
3. **Look at the test set once**, at the very end, with the configuration chosen by cross-validation.

As the model we use a support vector machine with an RBF kernel, `class_weight='balanced'` to counter the class imbalance, and UAR (`balanced_accuracy`) as the metric.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import balanced_accuracy_score, classification_report, ConfusionMatrixDisplay

y_train = train.emotion.to_numpy().astype(str)
y_test = test.emotion.to_numpy().astype(str)
spk_train = train.speaker.to_numpy()
spk_test = test.speaker.to_numpy()

import warnings
# speaker 8 has no 'disgust' utterance (Task 0.1): sklearn warns when that fold's UAR is computed over 6 classes only
warnings.filterwarnings('ignore', message='y_pred contains classes not in y_true')

def svm():
    return make_pipeline(SimpleImputer(), StandardScaler(), SVC(kernel='rbf', C=1.0, class_weight='balanced'))

## Task 4.1: Random vs. speaker-independent cross-validation
Predict first: which of the two cross-validations will give the higher UAR, and why?

In [ ]:
random_cv = StratifiedKFold(n_splits=6, shuffle=True, random_state=0)
speaker_cv = LeaveOneGroupOut()

for name, X in [('voice (Praat)', voice_train), ('eGeMAPS', egemaps_train)]:
    s_random = cross_val_score(svm(), X, y_train, cv=random_cv, scoring='balanced_accuracy')
    s_speaker = cross_val_score(svm(), X, y_train, groups=spk_train, cv=speaker_cv, scoring='balanced_accuracy')
    print(f'{name:15s} random CV: UAR = {s_random.mean():.3f} ± {s_random.std():.3f}    '
          f'speaker CV: UAR = {s_speaker.mean():.3f} ± {s_speaker.std():.3f}')

**Questions**
1. How large is the gap between random and speaker-independent CV? Which number would you report in a paper, and why?
2. Look at the standard deviation over the speaker folds. What does it tell you about how different speakers are?
3. In the PARLO corpus every speaker contributes up to 8 recordings (tasks). What happens if you split recordings randomly instead of speakers?

**Solution:**
1. Random CV is clearly better: about 0.65 vs. 0.57 UAR for the Praat features and 0.76 vs. 0.62 for eGeMAPS.
   With random folds the model has heard every speaker during training and partly recognises the speaker's voice and style.
   Only the speaker-independent number estimates the performance on new speakers, so that is the one to report.
2. The spread across speakers is large (about ±0.11 UAR): some actors are much easier to classify than others.
   With 6 to 10 speakers a single number hides this, so always report the spread.
3. The same patient would appear in training and test, the model can learn to recognise the person, and the results are inflated: speaker leakage.
   The PDC paper therefore recommends speaker-independent (and leave-one-site-out) validation.

## Task 4.2: Speaker normalisation
In Part 1 we saw that gender and speaker shift F0 more than emotion does.
**Speaker z-normalisation** standardises every feature *per speaker*: we fit a `StandardScaler` only on the recordings of that speaker.
Afterwards every feature describes how far an utterance deviates from the speaker's *own* average.
This needs no labels, so it is also allowed for test speakers, as long as several recordings per speaker are available.

In [ ]:
for name, Xtr in [('voice (Praat)', voice_train), ('eGeMAPS', egemaps_train)]:
    Xtr_norm = speaker_znorm(SimpleImputer().fit_transform(Xtr), spk_train)
    s = cross_val_score(svm(), Xtr_norm, y_train, groups=spk_train, cv=speaker_cv, scoring='balanced_accuracy')
    print(f'{name:15s} speaker CV, speaker-normalised: UAR = {s.mean():.3f} ± {s.std():.3f}')

**Questions**
1. How much does speaker normalisation help? Which kind of features profit most from it?
2. Speaker normalisation assumes that every speaker produces (roughly) the same mix of classes. Is this true in EmoDB? Is it true in the PARLO corpus, where each speaker has exactly one diagnosis?

**Solution:**
1. About 10 points of UAR for both sets (Praat 0.57 → 0.67, eGeMAPS 0.62 → 0.73), and the spread over speakers also shrinks.
   Absolute-level features profit most: mean F0, formants and intensity mainly encode the speaker (gender, vocal tract length, microphone distance), and after normalisation they encode "higher / louder / faster than this speaker usually is", which is exactly the emotion cue.
2. In EmoDB every actor speaks all emotions, so "deviation from the speaker's mean" is exactly the emotion effect.
   In the PARLO corpus all recordings of a speaker share **one** label.
   Normalising per speaker would remove exactly the information we want (a speaker's overall low F0 variability *is* the symptom).
   For a between-speaker task you can normalise per **task** or per **gender** over the training set instead, but not per speaker.

## Task 4.3: Choose a model by cross-validation, not on the test set
We tune the SVM's regularisation `C` and kernel width `gamma` with `GridSearchCV`, using the speaker folds and UAR, and we compare four feature sets.
Only the training set is used.

In [ ]:
feature_sets = {
    'voice (Praat, 15)': (voice_train, voice_test),
    'MFCC functionals (80)': (mfcc_train, mfcc_test),
    'eGeMAPS (88)': (egemaps_train, egemaps_test),
    'ComParE (6373)': (compare_train, compare_test),
}
param_grid = {'svc__C': [0.1, 1, 10, 100], 'svc__gamma': ['scale', 0.01, 0.001]}

searches = {}
for name, (Xtr, _) in feature_sets.items():
    search = GridSearchCV(svm(), param_grid, cv=speaker_cv, scoring='balanced_accuracy', n_jobs=-1)
    search.fit(Xtr, y_train, groups=spk_train)
    searches[name] = search
    print(f'{name:25s} best CV UAR = {search.best_score_:.3f}   {search.best_params_}')

## Task 4.4: The final test
Pick the feature set with the best cross-validation score.
`GridSearchCV` has already re-fitted the best configuration on the whole training set (`best_estimator_`).
Evaluate it **once** on the test speakers.

In [ ]:
best_name = max(searches, key=lambda n: searches[n].best_score_)
model = searches[best_name].best_estimator_
X_test = feature_sets[best_name][1]

y_pred = model.predict(X_test)
print(best_name, f'test UAR = {balanced_accuracy_score(y_test, y_pred):.3f}')
print(classification_report(y_test, y_pred, labels=EMOTIONS, digits=3))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=EMOTIONS, normalize='true',
                                        xticks_rotation=45, values_format='.2f', cmap='Blues');

## Task 4.5: Which features matter?
Permutation importance shuffles one feature at a time and measures how much the UAR drops.
We use the interpretable Praat set, fit on train and measured on test (the model selection is finished, so this no longer influences any decision).

In [ ]:
from sklearn.inspection import permutation_importance

voice_model = searches['voice (Praat, 15)'].best_estimator_
imp = permutation_importance(voice_model, voice_test, y_test, scoring='balanced_accuracy', n_repeats=20, random_state=0)
order = np.argsort(imp.importances_mean)
plt.figure(figsize=(8, 5))
plt.barh(voice_test.columns[order], imp.importances_mean[order], xerr=imp.importances_std[order])
plt.xlabel('drop in UAR when the feature is shuffled');

**Questions**
1. Which feature set won the cross-validation? Did more features help (ComParE vs. eGeMAPS)? Why could a huge feature set be a bad idea with 304 training utterances?
2. Is the test UAR close to the cross-validation UAR? If not, what could explain the difference?
3. Which emotions are confused most? Does this match your prediction from the feature profile in Part 2?
4. Which features are the most important? Are they the ones you expected from the emotion profiles?
5. Why was it important not to use the test set in Task 4.3?

**Solution:**
1. ComParE wins (about 0.68 vs. 0.63 for eGeMAPS, 0.61 for MFCCs and 0.59 for the Praat features), but with 70 times more features than eGeMAPS.
   With 6373 features and 304 samples the model can easily fit noise, and every extra choice in model selection makes the best cross-validation score more optimistic.
   Remember also that speaker-normalised eGeMAPS (Task 4.2) reached 0.73 with 88 interpretable features: the evaluation setup matters as much as the feature set.
2. Yes, the test UAR (about 0.73) is even a bit higher than the CV estimate (0.68).
   With only 4 test speakers, the test score itself has a large uncertainty: one unusual actor can move it by several points in either direction.
3. Happiness is mostly predicted as anger (recall below 0.2), the classic high-arousal confusion; fear and disgust also lose recall, mostly to anger.
   Sadness, neutral and boredom are recognised well.
4. Mean F0 is clearly the most important feature, followed by shimmer, CPPS, duration (speaking rate) and HNR.
   These are the arousal and voice quality cues from Parts 1 and 2: high, fast and tense for anger, breathy and slow for sadness.
   F1 and F2 hardly matter: they mainly depend on *what* is said, which is the same in every class.
5. Every look at the test set that influences a decision (feature set, C, gamma) turns the test set into a validation set.
   The reported number is then optimistically biased and no longer estimates the performance on new data.

# Part 5: Transfer to the PStA (5 min)

Discuss in your team and write down short answers. They are a good starting point for your first check-in.
1. Which of today's features would you expect to differ between HC, MCI and DEM, based on the literature summarised in the PStA slides (pauses, speech rate, prosodic variability, voice quality)?
2. Which PARLO task would you use for which feature (story reading, picture description, "pataka", recall)? Why should features from different tasks not be mixed in one normalisation?
3. Which confounders (age, gender, education, recording site) could a classifier pick up in the PARLO corpus, and how would you check for them?
4. EmoDB was easy: actors, exaggerated emotions, a studio. What will be harder with patients in memory clinics?

**Solution:**
1. Pause ratio and pause duration, speech rate (syllables or words per second), F0 variability (range, standard deviation) and, in later stages, voice quality (CPPS, HNR).
   Effects are expected to be much smaller than between acted emotions.
2. Timing features (pauses, rate) are most informative in open tasks (picture description, recall), where planning and word-finding are needed.
   Voice quality and articulation are more comparable in constrained tasks (reading, "pataka": diadochokinetic rate and regularity).
   Each task has its own typical values (reading is faster than recall), so normalising across tasks mixes task effects with speaker effects.
3. Gender (F0), age (voice quality, speech rate), education (vocabulary, reading fluency) and site (room, examiner, interventions).
   Check them by comparing their distribution per class, by predicting the confounder from your features, and by evaluating per subgroup or leave-one-site-out.
4. Natural, subtle effects; only one label per speaker, so no within-speaker contrasts; examiner speech in the recordings; heterogeneous MCI; fewer recordings per class; and the clinical cost of errors.

## Final questions
Answer in your own words, two to three sentences each.
1. What is the difference between jitter, shimmer, HNR and CPP, and which aspect of voice production does each one capture?
2. What are low-level descriptors and functionals, and why do we need functionals for a classical classifier?
3. Why does random cross-validation over-estimate the performance of a speech classifier, and how do you avoid it?
4. When is speaker normalisation a good idea, and when does it destroy the information you want to classify?

**Solution:**
1. Jitter: cycle-to-cycle variation of the period (frequency stability of the vocal folds).
   Shimmer: cycle-to-cycle variation of the amplitude (stability of closure and airflow).
   HNR: ratio of periodic to noise energy (breathiness, hoarseness).
   CPP: how prominent the dominant periodicity is in the cepstrum, a robust overall measure of dysphonia.
2. LLDs are frame-wise measurements (one value every 10 ms), and functionals (mean, std, percentiles, ...) summarise their contour over an utterance.
   Recordings have different lengths, but an SVM or random forest needs one fixed-length vector per sample.
3. Utterances of the same speaker are correlated (voice, style, setup), so a random split puts the same speaker in training and test and the model profits from recognising the speaker.
   Use group-wise splits (`LeaveOneGroupOut`, `GroupKFold`) with the speaker as group, and keep a speaker-disjoint test set.
4. It helps when every speaker produces several classes (emotions, or states over time), because then it removes speaker identity while keeping within-speaker contrasts.
   It hurts when the label is constant per speaker (diagnosis), because then the speaker's deviation from the population *is* the signal.